# Housekeeping

In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Input, Dense, Lambda
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split

from transformers import BertTokenizer, TFBertModel
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset, random_split
from torch import nn, optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report



In [ ]:
from google.colab import files

uploaded = files.upload()

Saving fake_job_postings.csv to fake_job_postings.csv


In [4]:
df = pd.read_csv("fake_job_postings.csv")
df.head(5)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [5]:
sum(df['fraudulent'] == 1)

866

# Data Cleaning

In [6]:
df.columns

Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='object')

In [7]:
import re
import string
import spacy
import pandas as pd

# Fill NA with empty space
df.fillna(' ', inplace=True)

# Concatenate all text features into one big text
df['text'] = df['title'] + " " + df['company_profile'] + " " + df['description'] + " " + df['requirements'] + " " + df['benefits']

# Define the clean_text function
def clean_text(text):
    '''Make text lowercase, remove text in square brackets, remove links, remove punctuation
    and remove words containing numbers.'''
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

# Apply the clean_text function to the 'text' column
df['text'] = df['text'].apply(lambda x: clean_text(x))

# Data Cleanups
df['text'] = df['text'].str.replace('\n', '')
df['text'] = df['text'].str.replace('\r', '')
df['text'] = df['text'].str.replace('\t', '')
df['text'] = df['text'].apply(lambda x: re.sub(r'[0-9]', '', x))
df['text'] = df['text'].apply(lambda x: re.sub(r'[/(){}\[\]\|@,;.:-]', ' ', x))
df['text'] = df['text'].apply(lambda s: s.lower() if type(s) == str else s)
df['text'] = df['text'].str.replace('  ', ' ')

# Remove Stop words
nlp = spacy.load("en_core_web_sm")
df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if nlp.vocab[word].is_stop == False]))

# Drop unnecessary columns
df = df[['text', 'fraudulent']]

# Display the modified DataFrame
print(df.head(3))



                                                text  fraudulent
0  marketing intern weve created groundbreaking a...           0
1  customer service cloud video production second...           0
2  commissioning machinery assistant cma valor se...           0


In [8]:
df.iloc[0]['text']

'marketing intern weve created groundbreaking awardwinning cooking site support connect celebrate home cooks need placewe editorial business engineering team focused technology find new better ways connect people specific food interests offer superb highly curated information food cooking attract talented home cooks contributors country publish wellknown professionals like mario batali gwyneth paltrow danny meyer partnerships foods market random named best food website james beard foundation iacp featured new york times npr pando daily techcrunch today showwere located chelsea new york city fastgrowing james beard awardwinning online food community crowdsourced curated recipe hub currently interviewing parttime unpaid interns work small team editors executives developers new york city headquartersreproducing andor repackaging existing content number partner sites huffington post yahoo buzzfeed content management systemsresearching blogs websites provisions affiliate programassisting da

# Modeling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size = 0.20, random_state = 1)


In [ ]:
X_train.tolist()

['business developer austria switzerland massive media social media company successful digital brands november massive media bought relaunched social discovery platform stepout enable members meet nearby people instantly million people joined sites web mobile functionwe’re normal website – we’re social community platform unified mission create unexpected ways online advertising change brand perception we’re growing rapidly variety european national accounts we’re looking experience selling online media campaigns multiple clients goes banner strong interest indepth understanding digital media landscape including emerging media social networking dedication willingness learn drive online advertising revenues integrated branding products netlog austria amp switzerland liaise new strategic revenue generating partners translate client marketing advertising objectives successful digital media strategies look digital order develop best possible campaign results customersskills experienceyou ho

In [10]:
checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [ ]:
x_train_tokenized = bert_tokenizer(X_train.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')


In [ ]:

x_test_tokenized = bert_tokenizer(X_test.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')


In [ ]:
# Implement default bert classification model from class

MAX_SEQUENCE_LENGTH = 128

def create_bert_classification_model(bert_model,
                                     num_train_layers=0,
                                     hidden_size = 200,
                                     dropout=0.3,
                                     learning_rate=0.00005):
    """
    Build a simple classification model with BERT. Use the Pooler Output for classification purposes
    """
    if num_train_layers == 0:
        # Freeze all layers of pre-trained BERT model
        bert_model.trainable = False

    elif num_train_layers == 12:
        # Train all layers of the BERT model
        bert_model.trainable = True

    else:
        # Restrict training to the num_train_layers outer transformer layers
        retrain_layers = []

        for retrain_layer_number in range(num_train_layers):

            layer_code = '_' + str(11 - retrain_layer_number)
            retrain_layers.append(layer_code)


        print('retrain layers: ', retrain_layers)

        for w in bert_model.weights:
            if not any([x in w.name for x in retrain_layers]):
                #print('freezing: ', w)
                w._trainable = False

    input_ids = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='input_ids_layer')
    token_type_ids = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='token_type_ids_layer')
    attention_mask = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='attention_mask_layer')

    bert_inputs = {'input_ids': input_ids,
                   'token_type_ids': token_type_ids,
                   'attention_mask': attention_mask}

    bert_out = bert_model(bert_inputs)

    pooler_token = bert_out[1]

    hidden = tf.keras.layers.Dense(hidden_size, activation='relu', name='hidden_layer')(pooler_token)


    hidden = tf.keras.layers.Dropout(dropout)(hidden)


    classification = tf.keras.layers.Dense(1, activation='sigmoid',name='classification_layer')(hidden)

    classification_model = tf.keras.Model(inputs=[input_ids, token_type_ids, attention_mask], outputs=[classification])

    classification_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                                 loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
                                 metrics='accuracy')

    return classification_model

In [ ]:
bert_model = TFBertModel.from_pretrained(checkpoint)
bert_classification_model = create_bert_classification_model(bert_model, num_train_layers=0)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [ ]:
bert_classification_model_history = bert_classification_model.fit(
    [x_train_tokenized.input_ids, x_train_tokenized.token_type_ids, x_train_tokenized.attention_mask],
    y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=3
)

Epoch 1/3
354/358 [============================>.] - ETA: 1s - loss: 0.2197 - accuracy: 0.9398

In [ ]:
len(X_test), sum(y_test)

In [ ]:
153/3576

In [ ]:
from sklearn.metrics import classification_report

# Assuming you have already trained your model and have predictions
y_pred = bert_classification_model.predict([x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask])
y_pred_classes = (y_pred > 0.5).astype(int)  # Assuming you have a binary classification task and using a threshold of 0.5

# Print classification report
print(classification_report(y_test, y_pred_classes))

# Data Balancing

In [ ]:
df_good_ads = df[df['fraudulent'] == 0]
df_fraud_ads = df[df['fraudulent'] == 1]

In [ ]:
len(df_fraud_ads)

# 1 to 1 Balancing

In [ ]:
# Randomly sample from df1
df_good_ads_sampled = df_good_ads.sample(n = 866, random_state=42)

# Concatenate
one_to_one_df = pd.concat([df_good_ads_sampled, df_fraud_ads], axis=0)

In [ ]:
one_to_one_df

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(one_to_one_df['text'], one_to_one_df['fraudulent'], test_size = 0.20, random_state = 1)


In [ ]:
checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

In [ ]:
# Tokenize
x_train_tokenized = bert_tokenizer(X_train.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')
x_test_tokenized = bert_tokenizer(X_test.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')

#let's get a fresh instance of the bert_model -- good practice
bert_model = TFBertModel.from_pretrained(checkpoint)
bert_classification_model = create_bert_classification_model(bert_model, num_train_layers=0)

In [ ]:
bert_classification_model_history = bert_classification_model.fit(
    [x_train_tokenized.input_ids, x_train_tokenized.token_type_ids, x_train_tokenized.attention_mask],
    y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=3
)

In [ ]:
y_pred_probabilities = bert_classification_model.predict(
    [x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask]
)

# Convert probabilities to predicted labels (0 or 1)
y_pred = np.argmax(y_pred_probabilities, axis=1)


report = classification_report(y_test, y_pred)
print("Classification Report:\n", report)


# 2 to 1 Balancing

In [ ]:
# Randomly sample from df1
df_good_ads_sampled = df_good_ads.sample(n = 866 * 2, random_state=42)

# Concatenate
two_to_one_df = pd.concat([df_good_ads_sampled, df_fraud_ads], axis=0)

In [ ]:
two_to_one_df

Try something

In [ ]:
# tokenize
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
bert_model = TFBertModel.from_pretrained('bert-base-cased')




In [ ]:
# Split into train test
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state = 1)

# Import BERT
checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

# Tokenize
x_train_tokenized = bert_tokenizer(X_train.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')
x_test_tokenized = bert_tokenizer(X_test.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')

#let's get a fresh instance of the bert_model -- good practice
bert_model = TFBertModel.from_pretrained(checkpoint)
bert_classification_model = create_bert_avg_model(bert_model)

bert_classification_model_history = bert_classification_model.fit(
    [x_train_tokenized.input_ids, x_train_tokenized.token_type_ids, x_train_tokenized.attention_mask],
    y_train,
    validation_data=([x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask], y_test),
    batch_size=32,
    epochs=2
)

In [ ]:
y_pred_probabilities = bert_classification_model.predict(
    [x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask]
)

# Convert probabilities to predicted labels (0 or 1)
y_pred = np.argmax(y_pred_probabilities, axis=1)

In [ ]:
report = classification_report(y_test, y_pred)
report

In [ ]:
print("Classification Report:\n", report)


# Miscellaneous

Bert Classification Model

In [ ]:
MAX_SEQUENCE_LENGTH = 128

def create_bert_classification_model(bert_model,
                                     num_train_layers=0,
                                     hidden_size = 200,
                                     dropout=0.3,
                                     learning_rate=0.00005):
    """
    Build a simple classification model with BERT. Use the Pooler Output for classification purposes
    """
    if num_train_layers == 0:
        # Freeze all layers of pre-trained BERT model
        bert_model.trainable = False

    elif num_train_layers == 12:
        # Train all layers of the BERT model
        bert_model.trainable = True

    else:
        # Restrict training to the num_train_layers outer transformer layers
        retrain_layers = []

        for retrain_layer_number in range(num_train_layers):

            layer_code = '_' + str(11 - retrain_layer_number)
            retrain_layers.append(layer_code)


        print('retrain layers: ', retrain_layers)

        for w in bert_model.weights:
            if not any([x in w.name for x in retrain_layers]):
                #print('freezing: ', w)
                w._trainable = False

    input_ids = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='input_ids_layer')
    token_type_ids = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='token_type_ids_layer')
    attention_mask = tf.keras.layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int64, name='attention_mask_layer')

    bert_inputs = {'input_ids': input_ids,
                   'token_type_ids': token_type_ids,
                   'attention_mask': attention_mask}

    bert_out = bert_model(bert_inputs)

    pooler_token = bert_out[1]
    #cls_token = bert_out[0][:, 0, :]

    hidden = tf.keras.layers.Dense(hidden_size, activation='relu', name='hidden_layer')(pooler_token)


    hidden = tf.keras.layers.Dropout(dropout)(hidden)


    classification = tf.keras.layers.Dense(1, activation='sigmoid',name='classification_layer')(hidden)

    classification_model = tf.keras.Model(inputs=[input_ids, token_type_ids, attention_mask], outputs=[classification])

    classification_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                                 loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
                                 metrics='accuracy')

    return classification_model


# Split into train test
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state = 1)

# Import BERT
checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

# Tokenize
x_train_tokenized = bert_tokenizer(X_train.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')
x_test_tokenized = bert_tokenizer(X_test.tolist(), max_length = 128, truncation = True, padding = 'max_length', return_tensors = 'tf')

#let's get a fresh instance of the bert_model -- good practice
bert_model = TFBertModel.from_pretrained(checkpoint)
bert_classification_model = create_bert_avg_model(bert_model)

bert_classification_model_history = bert_classification_model.fit(
    [x_train_tokenized.input_ids, x_train_tokenized.token_type_ids, x_train_tokenized.attention_mask],
    y_train,
    validation_data=([x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask], y_test),
    batch_size=32,
    epochs=2
)


y_pred_probabilities = bert_classification_model.predict(
    [x_test_tokenized.input_ids, x_test_tokenized.token_type_ids, x_test_tokenized.attention_mask]
)

# Convert probabilities to predicted labels (0 or 1)
y_pred = np.argmax(y_pred_probabilities, axis=1)


report = classification_report(y_test, y_pred)
print("Classification Report:\n", report)


#BERT Model Isolated

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Load BERT
checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

# Input layer
input_ids = Input(shape=(128,), dtype=tf.int32, name="input_ids")

# BERT outputs
bert_outputs = bert_model(input_ids)[1]  # Taking the pooled output

# Classification
output = Dense(1, activation='sigmoid')(bert_outputs)

# Run model
model = Model(inputs=input_ids, outputs=output)

# Compile
model.compile(optimizer=Adam(learning_rate=2e-5), loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Tokenize
max_length = 128
tokenized_test = bert_tokenizer(list(X_test), truncation=True, padding='max_length', max_length=max_length, return_tensors="tf")

# Predict
predictions = model.predict(tokenized_test['input_ids'])

# Adjust threshold
binary_predictions = (predictions > 0.5).astype(int)


accuracy = accuracy_score(y_test, binary_predictions)
report = classification_report(y_test, binary_predictions)

print("Accuracy:", accuracy)
print("Classification Report:\n", report)

# BERT Model with class weights

In [14]:
# Putting this here again for ease of use
df_good_ads = df[df['fraudulent'] == 0]
df_fraud_ads = df[df['fraudulent'] == 1]

# Randomly sample from df1
df_good_ads_sampled = df_good_ads.sample(n = 866 * 2, random_state=42)

# Concatenate
two_to_one_df = pd.concat([df_good_ads_sampled, df_fraud_ads], axis=0)



In [15]:
from sklearn.metrics import accuracy_score, classification_report

# Split into train test
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state = 1)

checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

# Tokenize
max_length = 128  # Adjust as needed based on your data
tokenized_train = bert_tokenizer(list(X_train), truncation=True, padding='max_length', max_length=max_length, return_tensors="tf")
tokenized_test = bert_tokenizer(list(X_test), truncation=True, padding='max_length', max_length=max_length, return_tensors="tf")

# Input layer
input_ids = Input(shape=(max_length,), dtype=tf.int32, name="input_ids")

# BERT outputs
bert_outputs = bert_model(input_ids)[1]

# Classification layer
output = Dense(1, activation='sigmoid')(bert_outputs)

# Build
model = Model(inputs=input_ids, outputs=output)

# Compile
model.compile(optimizer=Adam(learning_rate=2e-5), loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()

# Train the model with class weights
class_weights = {0: 1.0, 1: 10.0}
model.fit(tokenized_train['input_ids'], y_train, epochs=5, batch_size=32, class_weight=class_weights)

# Predict
predictions = model.predict(tokenized_test['input_ids'])

# Adjust treshold
threshold = 0.3
binary_predictions = (predictions > threshold).astype(int)

accuracy = accuracy_score(y_test, binary_predictions)
report = classification_report(y_test, binary_predictions)

print("Accuracy:", accuracy)
print("Classification Report:\n", report)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_ids (InputLayer)      [(None, 128)]             0         
                                                                 
 tf_bert_model_3 (TFBertMod  TFBaseModelOutputWithPo   108310272 
 el)                         olingAndCrossAttentions             
                             (last_hidden_state=(Non             
                             e, 128, 768),                       
                              pooler_output=(None, 7             
                             68),                                
                              past_key_values=None,              
                             hidden_states=None, att             
                             entions=None, cross_att             
                             entions=None)                       
                                                           

Run it with the whole dataset

In [13]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size = 0.20, random_state = 1)

checkpoint = 'bert-base-cased'
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = TFBertModel.from_pretrained(checkpoint)

# Tokenize
max_length = 128  # Adjust as needed based on your data
tokenized_train = bert_tokenizer(list(X_train), truncation=True, padding='max_length', max_length=max_length, return_tensors="tf")
tokenized_test = bert_tokenizer(list(X_test), truncation=True, padding='max_length', max_length=max_length, return_tensors="tf")

# Input layer
input_ids = Input(shape=(max_length,), dtype=tf.int32, name="input_ids")

# BERT outputs
bert_outputs = bert_model(input_ids)[1]

# Classification layer
output = Dense(1, activation='sigmoid')(bert_outputs)

# Build
model = Model(inputs=input_ids, outputs=output)

# Compile
model.compile(optimizer=Adam(learning_rate=2e-5), loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()

# Train the model with class weights
class_weights = {0: 1.0, 1: 10.0}
model.fit(tokenized_train['input_ids'], y_train, epochs=5, batch_size=32, class_weight=class_weights)

# Predict
predictions = model.predict(tokenized_test['input_ids'])

# Adjust treshold
threshold = 0.3
binary_predictions = (predictions > threshold).astype(int)

accuracy = accuracy_score(y_test, binary_predictions)
report = classification_report(y_test, binary_predictions)

print("Accuracy:", accuracy)
print("Classification Report:\n", report)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_ids (InputLayer)      [(None, 128)]             0         
                                                                 
 tf_bert_model_2 (TFBertMod  TFBaseModelOutputWithPo   108310272 
 el)                         olingAndCrossAttentions             
                             (last_hidden_state=(Non             
                             e, 128, 768),                       
                              pooler_output=(None, 7             
                             68),                                
                              past_key_values=None,              
                             hidden_states=None, att             
                             entions=None, cross_att             
                             entions=None)                       
                                                           